In [13]:
# 1. Pipeline up to step2 — needed for stim indices
from dutch_30_pipeline import Dutch30Pipeline
from dutch_30_feature_extractor import Dutch30FeatureExtractor
from dataset_config import Dutch30Config

config = Dutch30Config()
extractor = Dutch30FeatureExtractor(config=config)
pipeline = Dutch30Pipeline(extractor, config=config, use_wav2vec=False)
pipeline.step1_load_dutch30_data(patient_range=(21, 30))
pipeline.step2_split_by_instances(train_fraction=0.8)
pipeline.step3_load_channel_exclusions('channel_exclusions.json') 
pipeline.apply_channel_exclusions()

# (skip step3-5; no collapsed features)

# 2. Build minimal saved_models from MFA + your manner table
from run_pipeline import load_mfa_alignments
from collections import Counter
import numpy as np

MANNER = {
    # ---- Vowels (manner=0) ----
    # short
    'ɪ': 0, 'ɛ': 0, 'ɑ': 0, 'ɔ': 0, 'ʏ': 0, 'ə': 0, 'a': 0,
    # long / tense
    'i':  0, 'iː': 0, 'eː': 0, 'aː': 0, 'oː': 0,
    'uː': 0, 'yː': 0, 'øː': 0, 'u': 0, 'y': 0, 'e': 0, 'o': 0, 'ɔ̈':  0, 'ɛ̈':  0,
    # diphthongs
    'ɛi': 0, 'ɛj': 0, 'œy': 0, 'ɔu': 0, 'ɑu': 0, 'ɑi': 0, 'au': 0,
    'ui': 0, 'oi': 0, 'œ':  0,

    # ---- Stops / plosives (manner=1) ----
    'p': 1, 'b': 1, 't': 1, 'd': 1, 'k': 1, 'g': 1, 'ɡ': 1,
    'ʔ': 1, 'c':  1,   # palatal stop

    # ---- Fricatives (manner=2) ----
    'f': 2, 'v': 2, 's': 2, 'z': 2, 'x': 2, 'ɣ': 2, 'h': 2,
    'ʃ': 2, 'ʒ': 2, 'ç': 2, 'χ': 2,

    # ---- Nasals (manner=3) ----
    'm': 3, 'n': 3, 'ŋ': 3, 'ɲ': 3,

    # ---- Approximants / liquids (manner=4) ----
    'l': 4, 'r': 4, 'ʁ': 4, 'ɹ': 4, 'j': 4, 'ʋ': 4, 'w': 4, 'ɥ':  4,

    # ---- Affricates (treated as stops for manner) ----
    'ts': 1, 'tʃ': 1, 'dʒ': 1,
}

PLACE = {
    # ---- All vowels = 0 ----
    'ɪ': 0, 'ɛ': 0, 'ɑ': 0, 'ɔ': 0, 'ʏ': 0, 'ə': 0, 'a': 0,
    'i':  0, 'iː': 0, 'eː': 0, 'aː': 0, 'oː': 0,
    'uː': 0, 'yː': 0, 'øː': 0, 'u': 0, 'y': 0, 'e': 0, 'o': 0,
    'ɛi': 0, 'ɛj': 0, 'œy': 0, 'ɔu': 0, 'ɑu': 0, 'ɑi': 0, 'au': 0,
    'ui': 0, 'oi': 0, 'œ':  0, 'ɔ̈':  0, 'ɛ̈':  0,

    # ---- Bilabial (1): p b m ----
    'p': 1, 'b': 1, 'm': 1,

    # ---- Labiodental (2): f v ʋ ----
    'f': 2, 'v': 2, 'ʋ': 2, 'w': 2,

    # ---- Alveolar (3): t d s z n l r ts ----
    't': 3, 'd': 3, 's': 3, 'z': 3, 'n': 3, 'l': 3,
    'r': 3, 'ɹ': 3, 'ts': 3,

    # ---- Postalveolar (4): ʃ ʒ tʃ dʒ ----
    'ʃ': 4, 'ʒ': 4, 'tʃ': 4, 'dʒ': 4,

    # ---- Velar (5): k g x ɣ ŋ ----
    'k': 5, 'g': 5, 'ɡ': 5, 'x': 5, 'ɣ': 5, 'ŋ': 5,
    'χ': 5, 'ʁ': 5,

    # ---- Glottal (6): h ʔ ----
    'h': 6, 'ʔ': 6,

    # ---- Palatal (7): j ɲ ç ----
    'j': 7, 'ɲ': 7, 'ç': 7, 'c':  7, 'ɥ':  7,
}


def build_saved_state(pid, pipeline):
    wd = pipeline.split_result['word_segments_dict'][pid]
    mfa = load_mfa_alignments(pid)
    train_sent_ids = set()
    if 'train' in pipeline.split_result:
        for inst in pipeline.split_result['train'].get(pid, []):
            if isinstance(inst, dict) and 'sentence_idx' in inst:
                train_sent_ids.add(inst['sentence_idx'])
    if not train_sent_ids:
        all_real = [i for i, s in enumerate(wd['sentence_list'])
                    if isinstance(s, dict) and s.get('text')]
        train_sent_ids = set(i for i in all_real if i not in all_real[::6])

    # phoneme inventory from train sentences only (leak-safe)
    phons = []
    for sidx in train_sent_ids:
        if sidx in mfa:
            phons.extend(p['phone'] for p in mfa[sidx])
    inv = sorted(set(phons))
    cls_to_i = {ph: i for i, ph in enumerate(inv)}

    # bigram log-probs from train
    n = len(inv)
    bg = np.ones((n, n), dtype=np.float32)   # Laplace smoothing
    for sidx in train_sent_ids:
        if sidx not in mfa: continue
        seq = [cls_to_i[p['phone']] for p in mfa[sidx]
               if p['phone'] in cls_to_i]
        for a, b in zip(seq[:-1], seq[1:]):
            bg[a, b] += 1
    bg_lp = np.log(bg / bg.sum(axis=1, keepdims=True))

    # phone_to_manner array
    pm_arr = np.array([MANNER.get(ph, 0) for ph in inv], dtype=int)

    return {'cls_to_i': cls_to_i, 'bg_lp': bg_lp,
            'phone_to_manner': pm_arr}

# ---- Coverage check against the actual phonemes in your MFA output ----
def check_coverage(saved_models, manner=MANNER, place=PLACE):
    missing_manner = set()
    missing_place = set()
    for pid, st in saved_models.items():
        for ph in st['cls_to_i'].keys():
            if ph not in manner: missing_manner.add(ph)
            if ph not in place: missing_place.add(ph)
    if missing_manner:
        print(f"MISSING from MANNER: {sorted(missing_manner)}")
    if missing_place:
        print(f"MISSING from PLACE: {sorted(missing_place)}")
    if not missing_manner and not missing_place:
        print("All phonemes covered.")

saved_models = {pid: build_saved_state(pid, pipeline)
                for pid in ['P22', 'P23', 'P26', 'P29']}

Dutch30FeatureExtractor: Dutch30FeatureExtractor initialized:
Dutch30FeatureExtractor:   Data dir: C:\mozg\code\SingleWordProductionDutch\Dutch_30patients\raw
Dutch30FeatureExtractor:   Results dir: C:\mozg\code\SingleWordProductionDutch\results\dutch30
Dutch30FeatureExtractor:   Sampling rate: 1024 Hz
CustomBrainAudioDecoder: Initializing CustomBrainAudioDecoder with debug_mode=False
PhoneticDictionary: Initialized with DEBUG_MODE=False
PhoneticDictionary: Found 728 sentence-level entries
PhoneticDictionary: Added 1997 individual word entries
PhoneticDictionary: Skipped 0 sentences with unresolvable mismatches
UnifiedPhonemePipeline: Pipeline initialized: high_gamma, PCA=100, groups=False
PhoneticDictionary: Initialized with DEBUG_MODE=False
PhoneticDictionary: Found 728 sentence-level entries
PhoneticDictionary: Added 1997 individual word entries
PhoneticDictionary: Skipped 0 sentences with unresolvable mismatches
UnifiedPhonemePipeline: Pipeline initialized: high_gamma, groups=False

In [14]:
check_coverage(saved_models)

All phonemes covered.


In [15]:
# frame dataset, NO O tag, speech-only frames
import os, numpy as np
from scipy.signal import butter, sosfiltfilt, iirnotch, tf2sos, hilbert
from collections import defaultdict, Counter
from config import DUTCH_30_PATH
from run_pipeline import load_mfa_alignments
import torch

EEG_SR     = 1024
WIN_MS     = 30
SHIFT_MS   = 5
HG_LOW     = 70
HG_HIGH    = 170
NOTCH_HZ   = [50, 150]
FRAME_HZ   = int(1000 / SHIFT_MS)
WIN_SAMP   = int(EEG_SR * WIN_MS / 1000)
SHIFT_SAMP = int(EEG_SR * SHIFT_MS / 1000)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE = {DEVICE}")
if DEVICE == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  free memory: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

def _design_filters():
    sos_bp = butter(4, [HG_LOW, HG_HIGH], btype='bandpass', fs=EEG_SR, output='sos')
    sos_notches = []
    for f0 in NOTCH_HZ:
        b, a = iirnotch(f0, 30, EEG_SR)
        sos_notches.append(tf2sos(b, a))
    return sos_bp, sos_notches

_SOS_BP, _SOS_NOTCH = _design_filters()

def extract_hg_frames(eeg_slice):
    x = eeg_slice.astype(np.float64)
    for sos in _SOS_NOTCH:
        x = sosfiltfilt(sos, x, axis=0)
    x = sosfiltfilt(_SOS_BP, x, axis=0)
    env = np.abs(hilbert(x, axis=0))
    T = env.shape[0]
    n_frames = max(0, (T - WIN_SAMP) // SHIFT_SAMP + 1)
    out = np.zeros((n_frames, env.shape[1]), dtype=np.float32)
    for k in range(n_frames):
        s = k * SHIFT_SAMP
        out[k] = env[s:s + WIN_SAMP].mean(axis=0)
    return np.log1p(out)

def stack_context(X, K=5):
    T, C = X.shape
    pad = np.zeros((K, C), dtype=X.dtype)
    Xp = np.vstack([pad, X, pad])
    cols = [Xp[k:k + T] for k in range(2 * K + 1)]
    return np.concatenate(cols, axis=1)

def build_speech_only_labels(mfa_phones, n_frames):
    """Returns per-frame (bio_tag_str, phon_sym, keep_mask).
       keep_mask[i] = True if frame i lies inside some phoneme interval."""
    bio = [None] * n_frames
    phon = [None] * n_frames
    keep = np.zeros(n_frames, dtype=bool)
    for ph in mfa_phones:
        s_s, e_s = ph['start_s'], ph['end_s']
        sym = ph['phone']
        k_start = int(np.ceil((s_s * EEG_SR - WIN_SAMP / 2) / SHIFT_SAMP))
        k_end   = int(np.floor((e_s * EEG_SR - WIN_SAMP / 2) / SHIFT_SAMP))
        k_start = max(0, k_start)
        k_end   = min(n_frames - 1, k_end)
        if k_end < k_start:
            continue
        bio[k_start] = f'B-{sym}'; phon[k_start] = sym; keep[k_start] = True
        for k in range(k_start + 1, k_end + 1):
            bio[k] = f'I-{sym}'; phon[k] = sym; keep[k] = True
    return bio, phon, keep

def get_manner_table(saved_models, pid):
    state = saved_models[pid]
    cls_to_i = state['cls_to_i']
    pm_arr = np.asarray(state['phone_to_manner']).astype(int)
    i_to_cls = {v: k for k, v in cls_to_i.items()}
    return {i_to_cls[i]: int(pm_arr[i]) for i in range(len(pm_arr))}

try:
    PLACE_TABLE = PLACE
except NameError:
    PLACE_TABLE = {}

def build_frame_dataset_v2(pid, pipeline, saved_models, channel_mask=None):
    raw_eeg = np.load(os.path.join(DUTCH_30_PATH, 'raw', f'{pid}_sEEG.npy'))
    if raw_eeg.ndim == 2 and raw_eeg.shape[0] < raw_eeg.shape[1]:
        raw_eeg = raw_eeg.T
    word_data = pipeline.split_result['word_segments_dict'][pid]
    mfa = load_mfa_alignments(pid)
    if channel_mask is not None:
        raw_eeg = raw_eeg[:, channel_mask]

    # Use real split if available, fallback to every-6th
    test_sent_ids = set()
    if 'test' in pipeline.split_result:
        for inst in pipeline.split_result['test'].get(pid, []):
            if isinstance(inst, dict) and 'sentence_idx' in inst:
                test_sent_ids.add(inst['sentence_idx'])
    if not test_sent_ids:
        all_real = [i for i, s in enumerate(word_data['sentence_list'])
                    if isinstance(s, dict) and s.get('text')]
        test_sent_ids = set(all_real[::6])
        print(f"  [{pid}] WARNING: fallback split (every 6th sentence)")

    manner_map = get_manner_table(saved_models, pid)

    out = {'train': defaultdict(list), 'test': defaultdict(list)}
    n_used = n_skip = 0
    for sent_idx, sent in enumerate(word_data['sentence_list']):
        if not isinstance(sent, dict) or not sent.get('text'):
            continue
        if sent_idx not in mfa or not mfa[sent_idx]:
            n_skip += 1; continue
        s0 = sent['stim_start_idx']; s1 = sent['stim_end_idx']
        if s1 > raw_eeg.shape[0]:
            n_skip += 1; continue
        X = extract_hg_frames(raw_eeg[s0:s1])
        if X.shape[0] < 11:
            n_skip += 1; continue
        bio, phon, keep = build_speech_only_labels(mfa[sent_idx], X.shape[0])
        if keep.sum() < 11:
            n_skip += 1; continue
        Xs = stack_context(X, K=5)

        bio_arr = np.array([b for b, k in zip(bio, keep) if k])
        phon_arr = np.array([p for p, k in zip(phon, keep) if k])
        manner_arr = np.array([manner_map.get(p, -1) for p in phon_arr], dtype=np.int64)
        place_arr = np.array(
            [PLACE_TABLE.get(p, -1) for p in phon_arr], dtype=np.int64)
        sent_idx_arr = np.full(int(keep.sum()), sent_idx, dtype=int)

        split = 'test' if sent_idx in test_sent_ids else 'train'
        out[split]['X'].append(Xs[keep])
        out[split]['bio'].append(bio_arr)
        out[split]['phon'].append(phon_arr)
        out[split]['manner'].append(manner_arr)
        out[split]['place'].append(place_arr)
        out[split]['sent_idx'].append(sent_idx_arr)
        n_used += 1

    result = {}
    for split in ('train', 'test'):
        if not out[split]['X']:
            result[split] = None; continue
        result[split] = {k: (np.concatenate(v, axis=0) if isinstance(v[0], np.ndarray)
                             else v) for k, v in out[split].items()}
    print(f"  [{pid}] used={n_used} skipped={n_skip}  "
          f"train_frames={result['train']['X'].shape[0] if result['train'] else 0}  "
          f"test_frames={result['test']['X'].shape[0] if result['test'] else 0}")
    return result

TARGET_PIDS = ['P22', 'P23', 'P26', 'P29']
frame_datasets_v2 = {}
for pid in TARGET_PIDS:
    print(f"\nBuilding {pid}...")
    try:
        frame_datasets_v2[pid] = build_frame_dataset_v2(pid, pipeline, saved_models)
    except Exception as e:
        print(f"  [{pid}] FAILED: {type(e).__name__}: {e}")

for pid, d in frame_datasets_v2.items():
    if d.get('train') is None: continue
    bc = Counter(d['train']['bio'])
    nB = sum(v for k, v in bc.items() if k.startswith('B-'))
    nI = sum(v for k, v in bc.items() if k.startswith('I-'))
    print(f"\n{pid}: B={nB}  I={nI}  ratio I/B={nI/max(nB,1):.1f}")

DEVICE = cuda
  GPU: NVIDIA GeForce RTX 5070
  free memory: 11.5 GB

Building P22...
  [P22] WARNING: fallback split (every 6th sentence)
  [P22] used=96 skipped=4  train_frames=49420  test_frames=9896

Building P23...
  [P23] WARNING: fallback split (every 6th sentence)
  [P23] used=94 skipped=6  train_frames=61509  test_frames=11212

Building P26...
  [P26] WARNING: fallback split (every 6th sentence)
  [P26] used=75 skipped=25  train_frames=34792  test_frames=6886

Building P29...
  [P29] WARNING: fallback split (every 6th sentence)
  [P29] used=100 skipped=0  train_frames=53698  test_frames=9275

P22: B=2792  I=46628  ratio I/B=16.7

P23: B=2773  I=58736  ratio I/B=21.2

P26: B=2007  I=32785  ratio I/B=16.3

P29: B=2915  I=50783  ratio I/B=17.4


In [16]:
#BiLSTM emission backbone + CRF (no O)

import torch
import torch.nn as nn
import torch.nn.functional as F

def build_tag_index_noO(cls_to_i):
    tag_to_idx = {}
    idx_to_tag = []
    for ph, c in sorted(cls_to_i.items(), key=lambda kv: kv[1]):
        tag_to_idx[f'B-{ph}'] = len(idx_to_tag); idx_to_tag.append(f'B-{ph}')
        tag_to_idx[f'I-{ph}'] = len(idx_to_tag); idx_to_tag.append(f'I-{ph}')
    return tag_to_idx, idx_to_tag, len(idx_to_tag)

def build_transition_mask_noO(idx_to_tag):
    n = len(idx_to_tag)
    mask = torch.zeros(n, n)
    info = []
    for t in idx_to_tag:
        if t.startswith('B-'): info.append(('B', t[2:]))
        elif t.startswith('I-'): info.append(('I', t[2:]))
    for j, (kj, pj) in enumerate(info):
        if kj == 'I':
            for i, (ki, pi) in enumerate(info):
                if not ((ki == 'B' and pi == pj) or (ki == 'I' and pi == pj)):
                    mask[i, j] = float('-inf')
    return mask

def init_transition_matrix_noO(idx_to_tag, bg_lp=None, cls_to_i=None):
    n = len(idx_to_tag)
    T = torch.zeros(n, n)
    if bg_lp is None or cls_to_i is None:
        return T
    bg = torch.from_numpy(bg_lp).float()
    # B-X index for phoneme idx c: 2*c
    for ph_a, ca in cls_to_i.items():
        ba = 2 * ca
        for ph_b, cb in cls_to_i.items():
            bb = 2 * cb
            T[ba, bb] = bg[ca, cb] * 0.5
    return T


class LinearChainCRF(nn.Module):
    def __init__(self, n_tags, transition_mask, transition_init):
        super().__init__()
        self.n_tags = n_tags
        self.trans = nn.Parameter(transition_init.clone())
        self.start = nn.Parameter(torch.zeros(n_tags))
        self.end   = nn.Parameter(torch.zeros(n_tags))
        self.register_buffer('mask', transition_mask)

    def _masked_trans(self):
        return self.trans + self.mask

    def _forward_alg(self, emissions):
        trans = self._masked_trans()
        alpha = self.start + emissions[0]
        for t in range(1, emissions.size(0)):
            alpha = torch.logsumexp(alpha.unsqueeze(1) + trans, dim=0) + emissions[t]
        return torch.logsumexp(alpha + self.end, dim=0)

    def _score(self, emissions, tags):
        trans = self._masked_trans()
        score = self.start[tags[0]] + emissions[0, tags[0]]
        for t in range(1, emissions.size(0)):
            score = score + trans[tags[t-1], tags[t]] + emissions[t, tags[t]]
        return score + self.end[tags[-1]]

    def neg_log_likelihood(self, emissions, tags):
        return self._forward_alg(emissions) - self._score(emissions, tags)

    def viterbi(self, emissions):
        T_ = emissions.size(0)
        trans = self._masked_trans()
        bp = torch.zeros(T_, self.n_tags, dtype=torch.long, device=emissions.device)
        vit = self.start + emissions[0]
        for t in range(1, T_):
            scores = vit.unsqueeze(1) + trans
            vit, bp[t] = scores.max(dim=0)
            vit = vit + emissions[t]
        vit = vit + self.end
        last = int(vit.argmax().item())
        path = [last]
        for t in range(T_ - 1, 0, -1):
            last = int(bp[t, last].item())
            path.append(last)
        return list(reversed(path))


class BiLSTM_BIO_CRF(nn.Module):
    def __init__(self, n_in, n_phon, n_manner=5, n_place=8,
                 lstm_hidden=128, lstm_layers=2, lstm_dropout=0.3,
                 head_dropout=0.3, transition_mask=None, transition_init=None,
                 n_tags=None):
        super().__init__()
        self.n_phon = n_phon
        self.n_tags = n_tags
        self.proj = nn.Sequential(
            nn.Linear(n_in, lstm_hidden * 2), nn.GELU(),
            nn.Dropout(head_dropout),
        )
        self.lstm = nn.LSTM(input_size=lstm_hidden * 2,
                            hidden_size=lstm_hidden,
                            num_layers=lstm_layers,
                            dropout=lstm_dropout if lstm_layers > 1 else 0.0,
                            bidirectional=True, batch_first=False)
        self.drop_out = nn.Dropout(head_dropout)
        self.bio_head    = nn.Linear(lstm_hidden * 2, n_tags)
        self.bio_aux_head = nn.Linear(lstm_hidden * 2, n_tags)  # for direct CE
        self.manner_head = nn.Linear(lstm_hidden * 2, n_manner)
        self.place_head  = nn.Linear(lstm_hidden * 2, n_place)
        self.crf = LinearChainCRF(n_tags, transition_mask, transition_init)

    def encode(self, x):
        # x: (T, F) → (T, H)
        h = self.proj(x).unsqueeze(1)                # (T, 1, H)
        h, _ = self.lstm(h)
        h = self.drop_out(h.squeeze(1))              # (T, 2H)
        return h

    def loss(self, x, tags, manner, place,
             lam_manner=0.3, lam_place=0.1, lam_bio_ce=0.5,
             b_boost=0.0, ce_weights=None):
        h = self.encode(x)
        bio_em = self.bio_head(h)
        mn_em = self.manner_head(h)
        pl_em = self.place_head(h)

        # Direct CE on BIO tags (gives emission head an unambiguous signal)
        bio_aux = self.bio_aux_head(h)
        if ce_weights is not None:
            bio_ce = F.cross_entropy(bio_aux, tags, weight=ce_weights)
        else:
            bio_ce = F.cross_entropy(bio_aux, tags)

        # Optional B-boost (less critical now that O is gone)
        if b_boost != 0:
            b_idx = torch.arange(0, self.n_tags, 2, device=bio_em.device)
            bio_em_b = bio_em.clone()
            bio_em_b[:, b_idx] = bio_em_b[:, b_idx] + b_boost
        else:
            bio_em_b = bio_em
        crf_nll = self.crf.neg_log_likelihood(bio_em_b, tags) / x.size(0)

        valid_m = manner >= 0
        mn_loss = (F.cross_entropy(mn_em[valid_m], manner[valid_m])
                   if valid_m.sum() > 0 else torch.tensor(0., device=x.device))
        valid_p = place >= 0
        pl_loss = (F.cross_entropy(pl_em[valid_p], place[valid_p])
                   if valid_p.sum() > 0 else torch.tensor(0., device=x.device))

        total = (crf_nll + lam_bio_ce * bio_ce
                 + lam_manner * mn_loss + lam_place * pl_loss)
        return total, {'crf': float(crf_nll.item()),
                       'bio_ce': float(bio_ce.item()),
                       'mn': float(mn_loss.item()),
                       'pl': float(pl_loss.item())}

    @torch.no_grad()
    def decode(self, x):
        h = self.encode(x)
        return self.crf.viterbi(self.bio_head(h))


def make_model_v2(pid, frame_datasets_v2, saved_models,
                  lstm_hidden=128, lstm_layers=2, lstm_dropout=0.3,
                  head_dropout=0.3):
    fd = frame_datasets_v2[pid]
    state = saved_models[pid]
    cls_to_i = state['cls_to_i']
    bg_lp = state['bg_lp']
    pm_arr = np.asarray(state['phone_to_manner']).astype(int)
    n_in = fd['train']['X'].shape[1]
    n_phon = len(cls_to_i)
    n_manner = int(pm_arr.max()) + 1
    n_place = 8

    tag_to_idx, idx_to_tag, n_tags = build_tag_index_noO(cls_to_i)
    trans_mask = build_transition_mask_noO(idx_to_tag)
    trans_init = init_transition_matrix_noO(idx_to_tag, bg_lp, cls_to_i)

    model = BiLSTM_BIO_CRF(
        n_in=n_in, n_phon=n_phon, n_manner=n_manner, n_place=n_place,
        lstm_hidden=lstm_hidden, lstm_layers=lstm_layers,
        lstm_dropout=lstm_dropout, head_dropout=head_dropout,
        transition_mask=trans_mask, transition_init=trans_init, n_tags=n_tags)
    return model, tag_to_idx, idx_to_tag

for pid in TARGET_PIDS:
    if pid not in frame_datasets_v2 or frame_datasets_v2[pid]['train'] is None:
        continue
    m, _, _ = make_model_v2(pid, frame_datasets_v2, saved_models)
    print(f"{pid}: params={sum(p.numel() for p in m.parameters())/1e6:.2f}M  "
          f"n_tags={m.n_tags}")

P22: params=1.16M  n_tags=82
P23: params=1.21M  n_tags=82
P26: params=1.09M  n_tags=70
P29: params=1.08M  n_tags=78


In [27]:
# GPU training + mid-phoneme mixup

import time
import torch.optim as optim

EPOCHS         = 40
LR             = 2.5e-4
WEIGHT_DECAY   = 1e-3
EARLY_STOP_PATIENCE = 6      # stop if no improvement for 8 epochs
LAM_MANNER     = 0.3
LAM_PLACE      = 0.1
LAM_BIO_CE     = 0.5
B_BOOST        = 0.0      # O is gone; usually unneeded
GRAD_CLIP      = 5.0
MIN_SENT_FRAMES = 30

# augmentation
P_AUG          = 0.5      # prob of augmenting a sentence in a step
AUG_FRAC       = 0.2      # fraction of mid-phoneme frames perturbed
MIX_RATIO      = 0.8      # anchor weight
MIN_PHON_LEN_FOR_AUG = 6  # frames; phoneme must be at least this long


def split_into_sentences_v2(split_dict, tag_to_idx):
    sents = []
    sidx = split_dict['sent_idx']
    boundaries = np.where(np.diff(sidx, prepend=sidx[0] - 1) != 0)[0].tolist() \
                 + [len(sidx)]
    n_oov_total = 0
    for k in range(len(boundaries) - 1):
        s, e = boundaries[k], boundaries[k + 1]
        if e - s < MIN_SENT_FRAMES: continue
        bio_str_full = split_dict['bio'][s:e]
        # filter frames whose tag is OOV (test phoneme not in train vocab)
        keep_local = np.array([t in tag_to_idx for t in bio_str_full])
        n_oov_total += int((~keep_local).sum())
        if keep_local.sum() < MIN_SENT_FRAMES: continue

        bio_str = bio_str_full[keep_local]
        tags = np.array([tag_to_idx[t] for t in bio_str], dtype=np.int64)

        # mid_phoneme mask (same as before, on the kept slice)
        mid_mask = np.zeros(len(tags), dtype=bool)
        i = 0
        while i < len(bio_str):
            if bio_str[i].startswith('B-'):
                ph = bio_str[i][2:]
                j = i + 1
                while j < len(bio_str) and bio_str[j] == f'I-{ph}':
                    j += 1
                if (j - i) >= MIN_PHON_LEN_FOR_AUG:
                    margin = 2
                    mid_mask[i + margin: j - margin] = True
                i = j
            else:
                i += 1

        X_full      = split_dict['X'][s:e]
        manner_full = split_dict['manner'][s:e]
        place_full  = split_dict['place'][s:e]
        sents.append({
            'X':      torch.from_numpy(X_full[keep_local]).float(),
            'tags':   torch.from_numpy(tags),
            'manner': torch.from_numpy(manner_full[keep_local]),
            'place':  torch.from_numpy(place_full[keep_local]),
            'mid':    torch.from_numpy(mid_mask),
            'sent_idx': int(sidx[s]),
        })
    if n_oov_total > 0:
        print(f"  dropped {n_oov_total} frames with OOV phonemes")
    return sents

def fit_mu_sd(sents):
    Xall = torch.cat([s['X'] for s in sents], dim=0).numpy()
    return Xall.mean(0), Xall.std(0)

def standardize(sents, mu, sd):
    sd_safe = np.where(sd < 1e-6, 1.0, sd)
    mu_t = torch.from_numpy(mu).float()
    sd_t = torch.from_numpy(sd_safe).float()
    for s in sents:
        s['X'] = (s['X'] - mu_t) / sd_t

def to_device(sents, device):
    for s in sents:
        for k in ('X', 'tags', 'manner', 'place', 'mid'):
            s[k] = s[k].to(device)

def make_ce_weights(sents_tr, n_tags, device):
    """Class weights for the BIO direct-CE loss: inverse frequency, clamped."""
    cnt = torch.zeros(n_tags)
    for s in sents_tr:
        for t in s['tags'].cpu().tolist():
            cnt[t] += 1
    cnt = cnt.clamp(min=1.0)
    w = (cnt.sum() / (n_tags * cnt))
    w = w.clamp(min=0.2, max=5.0)
    return w.to(device)

def augment_sentence(s_anchor, partner_pool, p_aug, aug_frac, mix_ratio, rng):
    """Returns (X_maybe_augmented, tags_unchanged)."""
    if rng.random() > p_aug or s_anchor['mid'].sum() == 0:
        return s_anchor['X']
    mid_idx = s_anchor['mid'].nonzero(as_tuple=False).flatten()
    n_perturb = max(1, int(len(mid_idx) * aug_frac))
    perm = mid_idx[torch.randperm(len(mid_idx), device=mid_idx.device)[:n_perturb]]
    partner = partner_pool[rng.integers(0, len(partner_pool))]
    # pick random frames from partner (any frames, length-matched via wrap)
    if partner['X'].size(0) == 0:
        return s_anchor['X']
    p_idx = torch.randint(0, partner['X'].size(0), (n_perturb,),
                          device=partner['X'].device)
    X = s_anchor['X'].clone()
    X[perm] = mix_ratio * X[perm] + (1.0 - mix_ratio) * partner['X'][p_idx]
    return X

def evaluate_for_selection(model, sents_te, idx_to_tag):
    model.eval()
    correct = total = 0
    pred_symbols = set()
    n_pred = 0
    with torch.no_grad():
        for s in sents_te:
            path = model.decode(s['X'])
            tg = s['tags'].tolist()
            for p, t in zip(path, tg):
                correct += (p == t); total += 1
            for p in path:
                tag = idx_to_tag[int(p)]
                if tag.startswith('B-'):
                    pred_symbols.add(tag[2:])
                    n_pred += 1
    tag_acc = correct / max(total, 1)
    # Relaxed: 3 unique phonemes → factor 1.0 (was 5)
    diversity = min(1.0, len(pred_symbols) / 3.0)
    # Relaxed: 0.5× gold coverage already counts as full
    coverage_ratio = min(1.0, n_pred / max(1, total / 34))   # 34 = 2× phoneme len
    composite = tag_acc * diversity * coverage_ratio
    return composite, tag_acc, len(pred_symbols), n_pred

def train_v2(pid, frame_datasets_v2, saved_models, epochs=EPOCHS, **arch):
    fd = frame_datasets_v2[pid]
    if fd['train'] is None or fd['test'] is None:
        return None
    model, tag_to_idx, idx_to_tag = make_model_v2(
        pid, frame_datasets_v2, saved_models, **arch)
    model = model.to(DEVICE)

    sents_tr = split_into_sentences_v2(fd['train'], tag_to_idx)
    sents_te = split_into_sentences_v2(fd['test'],  tag_to_idx)
    if not sents_tr:
        return None
    mu, sd = fit_mu_sd(sents_tr)
    standardize(sents_tr, mu, sd); standardize(sents_te, mu, sd)
    to_device(sents_tr, DEVICE); to_device(sents_te, DEVICE)

    ce_weights = make_ce_weights(sents_tr, model.n_tags, DEVICE)

    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    rng = np.random.default_rng(0)
    best_acc, best_state = 0.0, None
    no_improve = 0
    stopped_early = False

    for ep in range(epochs):
        model.train()
        perm = rng.permutation(len(sents_tr))
        losses = {'total': 0, 'crf': 0, 'bio_ce': 0, 'mn': 0, 'pl': 0}
        for idx in perm:
            s = sents_tr[idx]
            X_aug = augment_sentence(s, sents_tr, P_AUG, AUG_FRAC, MIX_RATIO, rng)
            opt.zero_grad()
            loss, parts = model.loss(
                X_aug, s['tags'], s['manner'], s['place'],
                lam_manner=LAM_MANNER, lam_place=LAM_PLACE,
                lam_bio_ce=LAM_BIO_CE, b_boost=B_BOOST,
                ce_weights=ce_weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            losses['total'] += float(loss.item())
            for k in ('crf', 'bio_ce', 'mn', 'pl'):
                losses[k] += parts[k]
        sched.step()

        if (ep + 1) % 5 == 0 or ep == 0:
            score, tag_acc, n_uniq, n_pred = evaluate_for_selection(
                model, sents_te, idx_to_tag)
            if score > best_acc and ep >= 5:
                best_acc = score
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1

            n = len(sents_tr)
            print(f"  [{pid}] ep{ep+1:3d} "
                  f"loss={losses['total']/n:6.2f} crf={losses['crf']/n:5.2f} "
                  f"bio_ce={losses['bio_ce']/n:5.2f} "
                  f"mn={losses['mn']/n:.2f} pl={losses['pl']/n:.2f}  "
                  f"score={score:.3f} tag_acc={tag_acc:.3f} "
                  f"n_uniq={n_uniq} n_pred={n_pred} "
                  f"(best_score={best_acc:.3f})")

            if no_improve >= EARLY_STOP_PATIENCE:
                print(f"  [{pid}] early stopping at ep {ep+1}")
                stopped_early = True
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # Save to disk for inspection
    import os, pickle
    os.makedirs('bio_models', exist_ok=True)
    save_path = f'bio_models/{pid}_biocrf_v2.pkl'
    with open(save_path, 'wb') as f:
        pickle.dump({
            'state_dict': model.state_dict(),
            'mu': mu, 'sd': sd,
            'tag_to_idx': tag_to_idx, 'idx_to_tag': idx_to_tag,
            'arch': dict(lstm_hidden=128, lstm_layers=2,
                         lstm_dropout=0.3, head_dropout=0.3),
            'best_acc': best_acc,
            'pid': pid,
        }, f)
    print(f"  [{pid}] saved to {save_path}")

    return {
        'model': model, 'mu': mu, 'sd': sd,
        'tag_to_idx': tag_to_idx, 'idx_to_tag': idx_to_tag,
        'sents_tr': sents_tr, 'sents_te': sents_te,
        'best_acc': best_acc,
    }
    
bio_results_v2 = {}
for pid in TARGET_PIDS:
    if pid not in frame_datasets_v2 or frame_datasets_v2[pid]['train'] is None:
        continue
    print(f"\n=== Training {pid} ===")
    t0 = time.time()
    bio_results_v2[pid] = train_v2(
        pid, frame_datasets_v2, saved_models, epochs=EPOCHS,
        lstm_hidden=128, lstm_layers=2, lstm_dropout=0.3, head_dropout=0.3)
    if bio_results_v2[pid]:
        print(f"  [{pid}] done in {time.time() - t0:.1f}s")


=== Training P22 ===
  [P22] ep  1 loss=  4.10 crf= 1.32 bio_ce= 4.29 mn=1.57 pl=1.64  score=0.001 tag_acc=0.058 n_uniq=2 n_pred=6 (best_score=0.000)
  [P22] ep  5 loss=  2.97 crf= 0.41 bio_ce= 3.99 mn=1.43 pl=1.35  score=0.002 tag_acc=0.091 n_uniq=3 n_pred=5 (best_score=0.000)
  [P22] ep 10 loss=  2.55 crf= 0.34 bio_ce= 3.48 mn=1.17 pl=1.20  score=0.036 tag_acc=0.071 n_uniq=22 n_pred=145 (best_score=0.036)
  [P22] ep 15 loss=  1.93 crf= 0.25 bio_ce= 2.69 mn=0.78 pl=0.98  score=0.084 tag_acc=0.102 n_uniq=29 n_pred=240 (best_score=0.084)
  [P22] ep 20 loss=  1.43 crf= 0.19 bio_ce= 2.02 mn=0.51 pl=0.77  score=0.070 tag_acc=0.070 n_uniq=28 n_pred=297 (best_score=0.084)
  [P22] ep 25 loss=  1.14 crf= 0.17 bio_ce= 1.60 mn=0.38 pl=0.64  score=0.104 tag_acc=0.104 n_uniq=26 n_pred=322 (best_score=0.104)
  [P22] ep 30 loss=  0.99 crf= 0.15 bio_ce= 1.38 mn=0.31 pl=0.56  score=0.116 tag_acc=0.116 n_uniq=25 n_pred=322 (best_score=0.116)
  [P22] ep 35 loss=  0.92 crf= 0.14 bio_ce= 1.29 mn=0.28 pl=

In [28]:
# ============================================================
# Adapter: BIO-CRF results → pipeline.patient_results format
# ============================================================
from e2e_brain_decoder import show_matched_sequences_with_times

FRAME_SHIFT_S = 5e-3   # 200 Hz

def collapse_bio_to_segments(bio_tag_strs):
    """Given a list of BIO tag strings (no O in v2), return parallel lists
    of (phoneme_symbol, start_frame_idx, end_frame_idx_exclusive)."""
    phons, starts, ends = [], [], []
    i = 0
    n = len(bio_tag_strs)
    while i < n:
        t = bio_tag_strs[i]
        if t.startswith('B-'):
            ph = t[2:]
            j = i + 1
            while j < n and bio_tag_strs[j] == f'I-{ph}':
                j += 1
            phons.append(ph); starts.append(i); ends.append(j)
            i = j
        elif t.startswith('I-'):
            # orphan I (shouldn't happen with structural mask, but be defensive)
            ph = t[2:]
            j = i + 1
            while j < n and bio_tag_strs[j] == f'I-{ph}':
                j += 1
            phons.append(ph); starts.append(i); ends.append(j)
            i = j
        else:
            i += 1
    return phons, starts, ends


def bio_results_to_patient_results(pid, bio_results_v2, frame_shift_s=FRAME_SHIFT_S):
    """Build the dict structure that show_matched_sequences_with_times reads."""
    res = bio_results_v2[pid]
    model = res['model']; model.eval()
    idx_to_tag = res['idx_to_tag']
    sents_te = res['sents_te']

    true_labels = []
    predictions = []
    true_sentence_ids = []
    pred_sentence_ids = []
    true_segments = []
    pred_segments = []

    with torch.no_grad():
        for s in sents_te:
            path = model.decode(s['X'])
            pred_tag_strs = [idx_to_tag[int(t)] for t in path]
            gold_tag_strs = [idx_to_tag[int(t)] for t in s['tags'].tolist()]

            pp, ps, pe = collapse_bio_to_segments(pred_tag_strs)
            gp, gs, ge = collapse_bio_to_segments(gold_tag_strs)

            sid = s['sent_idx']
            for ph, fa, fb in zip(pp, ps, pe):
                predictions.append(ph)
                pred_sentence_ids.append(sid)
                pred_segments.append((fa * frame_shift_s, fb * frame_shift_s))
            for ph, fa, fb in zip(gp, gs, ge):
                true_labels.append(ph)
                true_sentence_ids.append(sid)
                true_segments.append((fa * frame_shift_s, fb * frame_shift_s))

    # Compute concatenated edit distance / PER (the viewer wants real numbers)
    from e2e_brain_decoder import edit_distance
    true_arr = np.array(true_labels)
    pred_arr = np.array(predictions)
    ed = edit_distance(list(true_arr), list(pred_arr))
    per = ed / max(len(true_arr), 1)
    acc = float((true_arr[:min(len(true_arr), len(pred_arr))]
                 == pred_arr[:min(len(true_arr), len(pred_arr))]).mean()) \
          if len(true_arr) and len(pred_arr) else 0.0

    return {
        'true_labels':       true_arr,
        'predictions':       pred_arr,
        'true_sentence_ids': np.array(true_sentence_ids),
        'pred_sentence_ids': np.array(pred_sentence_ids),
        'true_segments':     true_segments,
        'pred_segments':     pred_segments,
        'accuracy':          acc,
        'edit_distance':     ed,
        'per':               per,
        'n_test':            len(true_labels),
        'n_pred':            len(predictions),
    }


# Build sentence_texts dict for nicer display
def build_sentence_texts(pipeline, pid):
    wd = pipeline.split_result['word_segments_dict'][pid]
    out = {}
    for i, s in enumerate(wd['sentence_list']):
        if isinstance(s, dict) and s.get('text'):
            out[i] = s['text']
    return out


# Wire into pipeline.patient_results and call the viewer
if not hasattr(pipeline, 'patient_results') or pipeline.patient_results is None:
    pipeline.patient_results = {}

for pid in bio_results_v2:
    pipeline.patient_results[pid] = bio_results_to_patient_results(pid, bio_results_v2)


import importlib, e2e_brain_decoder
importlib.reload(e2e_brain_decoder)
from e2e_brain_decoder import show_matched_sequences_with_times

# Now call the viewer
for pid in sorted(bio_results_v2.keys()):
    print(f"\n========== {pid} ==========")
    show_matched_sequences_with_times(
        pipeline, pid,
        max_per_line=30,
        time_align_tol_s=0.05,         # 50 ms tolerance for column alignment
        collapse_repeats=False,         # BIO already segmented; don't collapse twice
    )


========== P22 ==========

  P22  acc=5.33%  lift=1.76×  (33 classes)   edit=418  PER=75.04%
  40 matched n-grams: 35 of length ≥2  +  5 exact 1-grams  (shift tolerance: 2-gram±5, 3-gram±10, 4-gram±20)
  Coverage: 93/557 phonemes (16.7% in matched n-grams)




========== P23 ==========

  P23  acc=6.67%  lift=2.20×  (33 classes)   edit=394  PER=76.36%
  38 matched n-grams: 30 of length ≥2  +  8 exact 1-grams  (shift tolerance: 2-gram±5, 3-gram±10, 4-gram±20)
  Coverage: 76/516 phonemes (14.7% in matched n-grams)




========== P26 ==========

  P26  acc=4.13%  lift=1.16×  (28 classes)   edit=290  PER=75.92%
  25 matched n-grams: 20 of length ≥2  +  5 exact 1-grams  (shift tolerance: 2-gram±5, 3-gram±10, 4-gram±20)
  Coverage: 55/382 phonemes (14.4% in matched n-grams)




========== P29 ==========

  P29  acc=7.31%  lift=2.48×  (34 classes)   edit=382  PER=75.94%
  33 matched n-grams: 29 of length ≥2  +  4 exact 1-grams  (shift tolerance: 2-gram±5, 3-gram±10, 4-gram±20)
  Coverage: 66/503 phonemes (13.1% in matched n-grams)

